In [1]:
from nbtools import NotebookCache, show

# results and prints live on disk, so they survive kernel restarts; one folder per notebook
cache = NotebookCache('../cache/flow_matching_namespace')

# Flow matching on the robot (namespace + cache)

Redo of `flow_matching.ipynb` from step 7, the robot half, in `act_namespace.ipynb`'s style: a bundle read later is a **namespace** (`data.chunks.train`, `stats.action.std`), a small return unpacked right away is a **tuple**, slow loads are `@cache`d. Steps 0-6 (the toy) stay in `flow_matching.ipynb`.

## The robot: T2-fixed

Two mirrored bins, the cube always at the table centre, and the expert flips a coin for the bin each demo. 100 demos with half shake.

### step 7 — the data


In [2]:
import json
from pathlib import Path
from types import SimpleNamespace as namespace

import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from numpy.typing import NDArray

np.set_printoptions(precision=3, suppress=True)
REPO = Path.cwd().parent
ROOT = REPO / 'data/robosuite/pick_place_two_bins_fixed_noise0025_n100'

In [3]:
@cache
def load_table():
    # straight from the table, no video decode
    ds = LeRobotDataset('dream_robot/pick_place_two_bins', root=ROOT)
    return (
        np.asarray(ds.hf_dataset['observation.state'], dtype=np.float32),  # state [N, 8]
        np.asarray(ds.hf_dataset['action'], dtype=np.float32),  # action [N, 8]
        np.asarray(ds.hf_dataset['episode_index'], dtype=np.int64),  # episode [N], which demo
    )

load_table: cached, 1 call(s) stored


In [4]:
state, action, episode = load_table()  # three arrays, unpacked once: a tuple

# T2-fixed: the cube always starts at the table centre, so the scene never says which bin.
# The expert's coin isn't in the dataset; record.py logged it per recording attempt
attempts = [a for a in json.load(open(ROOT / 'bin_choices.json'))['attempts'] if a['kept']]
bin_of = np.array([a['target_bin'] for a in attempts])  # [100], episode e = the e-th kept attempt

# hold out 10 whole demos, 5 per bin: a plain random 10 came out 9 right / 1 left
rng = np.random.default_rng(0)
val_eps = np.concatenate(
    [rng.choice(np.flatnonzero(bin_of == b), 5, replace=False) for b in ['left', 'right']]
)
is_val = np.isin(episode, val_eps)  # [N]
show(
    state.shape,
    episode.max() + 1,
    bin_of.shape,
    (bin_of == 'left').sum(),
    (bin_of == 'right').sum(),
)
show(val_eps, (~is_val).sum(), is_val.sum())

state.shape = (21741, 8)
episode.max() + 1 = np.int64(100)
bin_of.shape = (100,)
(bin_of == 'left').sum() = np.int64(50)
(bin_of == 'right').sum() = np.int64(50)
val_eps = array([59, 49, 27, 31, 79, 74, 48, 60, 87, 99])
(~is_val).sum() = np.int64(19555)
is_val.sum() = np.int64(2186)


### step 8 — both cameras into RAM (~1 min the first time, cached after)


In [5]:
import torch
from torch import Tensor

In [6]:
@cache
def load_images():
    # every frame decoded from both videos once, kept as uint8 (~2.1 GB)
    ds = LeRobotDataset('dream_robot/pick_place_two_bins', root=ROOT)
    top = np.zeros((len(ds), 3, 128, 128), dtype=np.uint8)  # [N, 3, H, W]
    wrist = np.zeros_like(top)
    for i in range(len(ds)):
        item = ds[i]
        top[i] = (item['observation.images.top'] * 255).round().to(torch.uint8).numpy()
        wrist[i] = (item['observation.images.wrist'] * 255).round().to(torch.uint8).numpy()
    return top, wrist

load_images: cached, 1 call(s) stored


In [7]:
top, wrist = load_images()  # two arrays, unpacked at once: a tuple
show(top.shape, wrist.shape, f'{(top.nbytes + wrist.nbytes) / 1e9:.1f} GB')

top.shape = (21741, 3, 128, 128)
wrist.shape = (21741, 3, 128, 128)
f'{(top.nbytes + wrist.nbytes) / 1e9:.1f} GB' = '2.1 GB'


### step 9 — chunk targets (K = 16), everything to the GPU, normalisation stats

`make_chunks` as act step 1. Then `data` and `stats` as act step 4: nested namespaces, each field a train/val pair or a mean/std pair.


In [8]:
def make_chunks(action: NDArray[np.float32], episode: NDArray[np.int64], K: int):
    # demo e's last row is where its run of rows ends, since demos are stored in order
    assert (np.diff(episode) >= 0).all(), 'demos not stored in order'
    last = np.searchsorted(episode, episode, side='right') - 1  # [N]
    idx = np.arange(len(action))[:, None] + np.arange(K)  # [N, K], rows t .. t+K-1
    pad = idx > last[:, None]  # [N, K], True where the chunk runs past its demo's end
    chunks = action[np.minimum(idx, last[:, None])]  # [N, K, 8], past the end: the final action
    return chunks, pad


K = 16  # chunk length, the repo's flow-matching value: 16 steps = 0.5 s (ACT used 50)
chunks, pad = make_chunks(action, episode, K)
show(chunks.shape, pad.any(1).sum())

chunks.shape = (21741, 16, 8)
pad.any(1).sum() = np.int64(1500)


In [9]:
@lambda f: f()
def probe():
    last = np.searchsorted(episode, episode, side='right') - 1
    idx = np.arange(len(action))[:, None] + np.arange(K)
    show(
        last.shape,
        last[208:215],
        idx[195:200],
        pad[195:200],
        np.minimum(idx, last[:, None])[195:200],
    )

last.shape = (21741,)
last[208:215] = array([211, 211, 211, 211, 424, 424, 424])
idx[195:200] =
array([[195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207,
        208, 209, 210],
       [196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208,
        209, 210, 211],
       [197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209,
        210, 211, 212],
       [198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210,
        211, 212, 213],
       [199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211,
        212, 213, 214]])
pad[195:200] =
array([[False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False],
       [False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False],
       [False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False,  True],
       [False, F

In [10]:
dev = 'cuda'


def split(x: NDArray) -> namespace:
    # train/val by episode, each on the GPU; images stay uint8 until a batch is drawn
    return namespace(train=torch.tensor(x[~is_val]).to(dev), val=torch.tensor(x[is_val]).to(dev))


def mean_std(x: Tensor, dims: tuple[int, ...]) -> namespace:
    # keepdim leaves a 1 for each reduced dim, so the stats broadcast straight back over x
    return namespace(mean=x.mean(dims, keepdim=True), std=x.std(dims, keepdim=True))


data = namespace(
    top=split(top),
    wrist=split(wrist),
    state=split(state),
    chunks=split(chunks),
    pad=split(pad),
)

# from train only, as bc_2 / act. Action stats from single actions: a chunk is made of the same
# actions, so they apply to every step of it. State and action [1, 8]; images [1, 3, 1, 1],
# from every 20th frame
stats = namespace(
    state=mean_std(data.state.train, (0,)),
    action=mean_std(data.chunks.train[:, 0], (0,)),
    top=mean_std(data.top.train[::20].float() / 255, (0, 2, 3)),
    wrist=mean_std(data.wrist.train[::20].float() / 255, (0, 2, 3)),
)
show(data.chunks.train.shape, data.chunks.val.shape, stats.action.std)
show(f'GPU {torch.cuda.memory_allocated() / 1e9:.1f} GB')

data.chunks.train.shape = torch.Size([19555, 16, 8])
data.chunks.val.shape = torch.Size([2186, 16, 8])
stats.action.std =
tensor([[0.1861, 0.1934, 0.0519, 0.1271, 0.0828, 0.1632, 0.2331, 0.4805]],
       device='cuda:0')
f'GPU {torch.cuda.memory_allocated() / 1e9:.1f} GB' = 'GPU 2.1 GB'


In [11]:
@lambda f: f()
def probe():
    stats_shapes = [(k, getattr(stats, k).mean.shape) for k in vars(stats)]
    show(
        stats_shapes,
    )

stats_shapes = [('state', torch.Size([1, 8])), ('action', torch.Size([1, 8])), ('top', torch.Size([1, 3, 1, 1])), ('wrist', torch.Size([1, 3, 1, 1]))]


### step 10 — BC's image encoder, as before


In [12]:
from torch import nn


# bc_2's image encoder, unchanged (and act step 5): conv stack (bc_2 step 9) without its last
# ReLU, then spatial softmax (bc_2 step 15)
def make_conv() -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(3, 32, 5, 2, 2),  # -> [B, 32, 64, 64]
        nn.ReLU(),
        nn.Conv2d(32, 64, 3, 2, 1),  # -> [B, 64, 32, 32]
        nn.ReLU(),
        nn.Conv2d(64, 128, 3, 2, 1),  # -> [B, 128, 16, 16]
        nn.ReLU(),
        nn.Conv2d(128, 64, 3, 1, 1),  # -> [B, 64, 16, 16], no ReLU: the softmax wants negatives
    )


class SpatialSoftmax(nn.Module):
    cell_x: Tensor  # buffers, declared so the checker doesn't read them as Tensor | Module
    cell_y: Tensor

    def __init__(self, h: int, w: int):
        super().__init__()
        self.log_t = nn.Parameter(torch.zeros(()))  # learned temperature, T = exp(log_t) > 0
        self.register_buffer('cell_x', torch.linspace(-1, 1, w).repeat(h))  # [H*W]
        self.register_buffer('cell_y', torch.linspace(-1, 1, h).repeat_interleave(w))  # [H*W]

    def forward(self, f: Tensor) -> Tensor:
        # f: [B, C, H, W] feature maps
        # out: [B, 2C], an (x, y) per map
        B, C, H, W = f.shape
        p = torch.softmax(f.reshape(B, C, H * W) / self.log_t.exp(), dim=-1)  # [B, C, H*W]
        return torch.stack([p @ self.cell_x, p @ self.cell_y], dim=-1).reshape(B, 2 * C)

#### Spatial Softmax walkthrough


In [13]:
@lambda f: f()  # spatial softmax
def probe():
    h, w = 4, 4
    cell_x = torch.linspace(-1, 1, w).repeat(h)
    show(cell_x, cell_x.shape, cell_x.reshape(h, w))
    cell_y = torch.linspace(-1, 1, h).repeat_interleave(w)
    show(cell_y, cell_y.shape, cell_y.reshape(h, w))

    B, C, H, W = 1, 2, 4, 4
    f = torch.arange(B * C * H * W).reshape(B, C, H, W)
    show(f, f.reshape(B, C, H * W))
    T = 0.5  # peeky
    p = torch.softmax(f.reshape(B, C, H * W) / T, dim=-1)
    show(p.shape, T, p)
    T = 10  # uniform
    p = torch.softmax(f.reshape(B, C, H * W) / T, dim=-1)
    show(T, p)
    print('3 channels, random f')
    B, C, H, W = 1, 3, 4, 4
    f = torch.randn(B * C * H * W).reshape(B, C, H, W)
    p = torch.softmax(f.reshape(B, C, H * W) / T, dim=-1)
    show(p.shape)
    x = p @ cell_x
    show(x.shape, x)
    y = p @ cell_y
    show(y)
    sr = torch.stack([x, y], -1)
    show(sr, sr.shape)
    out = sr.reshape(B, 2 * C)
    show(out, out.shape)

cell_x =
tensor([-1.0000, -0.3333,  0.3333,  1.0000, -1.0000, -0.3333,  0.3333,  1.0000,
        -1.0000, -0.3333,  0.3333,  1.0000, -1.0000, -0.3333,  0.3333,  1.0000])
cell_x.shape = torch.Size([16])
cell_x.reshape(h, w) =
tensor([[-1.0000, -0.3333,  0.3333,  1.0000],
        [-1.0000, -0.3333,  0.3333,  1.0000],
        [-1.0000, -0.3333,  0.3333,  1.0000],
        [-1.0000, -0.3333,  0.3333,  1.0000]])
cell_y =
tensor([-1.0000, -1.0000, -1.0000, -1.0000, -0.3333, -0.3333, -0.3333, -0.3333,
         0.3333,  0.3333,  0.3333,  0.3333,  1.0000,  1.0000,  1.0000,  1.0000])
cell_y.shape = torch.Size([16])
cell_y.reshape(h, w) =
tensor([[-1.0000, -1.0000, -1.0000, -1.0000],
        [-0.3333, -0.3333, -0.3333, -0.3333],
        [ 0.3333,  0.3333,  0.3333,  0.3333],
        [ 1.0000,  1.0000,  1.0000,  1.0000]])
f =
tensor([[[[ 0,  1,  2,  3],
          [ 4,  5,  6,  7],
          [ 8,  9, 10, 11],
          [12, 13, 14, 15]],

         [[16, 17, 18, 19],
          [20, 21, 22, 23],
      

In [14]:
enc = nn.Sequential(make_conv(), SpatialSoftmax(16, 16)).to(dev)
x = data.top.train[:4].float() / 255  # [4, 3, 128, 128]
points = enc((x - stats.top.mean) / stats.top.std)  # [4, 128]: 64 maps -> 64 (x, y) points
show(x.shape, points.shape)

x.shape = torch.Size([4, 3, 128, 128])
points.shape = torch.Size([4, 128])


In [15]:
@lambda f: f()
def probe():
    conv = make_conv().to(dev)
    x_norm = (x - stats.top.mean) / stats.top.std
    conv_out = conv(x_norm)
    show(conv_out.shape)
    ss = SpatialSoftmax(16, 16).to(dev)
    points = ss(conv_out)
    show(points.shape)

conv_out.shape = torch.Size([4, 64, 16, 16])
points.shape = torch.Size([4, 128])


### step 11 — `VelocityNet`: (scene, noisy chunk, t) → speed of every number in the chunk

The toy's `cnet` (`flow_matching.ipynb` step 5), scaled up: the noisy answer is now a whole chunk `[K, 8]`, the scene is two cameras + state. One encoder per camera as named attributes, as act's `ChunkNet`.


In [16]:
class VelocityNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.top = nn.Sequential(make_conv(), SpatialSoftmax(16, 16))
        self.wrist = nn.Sequential(make_conv(), SpatialSoftmax(16, 16))
        n_in = 128 * 2 + 8 + K * 8 + 1  # top, wrist, state, the noisy chunk flattened, t
        self.mlp = nn.Sequential(
            nn.Linear(n_in, 512),
            nn.SiLU(),
            nn.Linear(512, 512),
            nn.SiLU(),
            nn.Linear(512, K * 8),
        )

    def forward(self, top: Tensor, wrist: Tensor, state: Tensor, xt: Tensor, t: Tensor) -> Tensor:
        # top, wrist: [B, 3, 128, 128] floats in [0, 1]
        # state: [B, 8] radians
        # xt: [B, K, 8] the noisy chunk at time t, in std units of action
        # t: [B] in [0, 1]
        # out: [B, K, 8] the predicted speed of every number in the chunk, std units per unit t
        f_top = self.top((top - stats.top.mean) / stats.top.std)  # [B, 128]
        f_wrist = self.wrist((wrist - stats.wrist.mean) / stats.wrist.std)  # [B, 128]
        st = (state - stats.state.mean) / stats.state.std  # [B, 8]
        h = torch.cat([f_top, f_wrist, st, xt.flatten(1), t[:, None]], dim=-1)  # [B, 393]
        return self.mlp(h).reshape(-1, K, 8)

In [17]:
vnet = VelocityNet().to(dev)
i = torch.arange(4, device=dev)
v = vnet(
    data.top.train[i].float() / 255,
    data.wrist.train[i].float() / 255,
    data.state.train[i],
    torch.randn(4, K, 8, device=dev),  # a noisy chunk: pure noise, as at t = 0
    torch.rand(4, device=dev),
)
show(sum(p.numel() for p in vnet.parameters()), v.shape)

sum(p.numel() for p in vnet.parameters()) = 867202
v.shape = torch.Size([4, 16, 8])


In [18]:
@lambda f: f()
def probe():
    n_in = 128 * 2 + 8 + K * 8 + 1
    show(n_in)

n_in = 393


### step 12 — train it (~4 min the first time, cached after)

The toy's training loop (`flow_matching.ipynb` step 5) on robot batches, plus BC's random shift and ACT's padding mask. 10k steps: the data is 4.6× T1's.

`get_batch` as act step 7; its train/val argument is `part`, since `split` is step 9's function.


In [19]:
from typing import Literal, Protocol

import torch.nn.functional as F


def random_shift(x: Tensor, pad: int = 4) -> Tensor:
    # as bc_2 step 30
    # x: [B, 3, H, W]
    # out: [B, 3, H, W], the whole batch moved by up to pad px each way, edge pixels repeated
    B, C, H, W = x.shape
    x = F.pad(x, (pad, pad, pad, pad), mode='replicate')  # [B, 3, H + 2p, W + 2p]
    dy, dx = (int(d) for d in torch.randint(0, 2 * pad + 1, (2,)))
    return x[:, :, dy : dy + H, dx : dx + W]


class Batch(Protocol):  # what get_batch returns: a namespace with these fields
    top: Tensor  # [B, 3, 128, 128] floats in [0, 1]
    wrist: Tensor  # [B, 3, 128, 128] floats in [0, 1]
    state: Tensor  # [B, 8] radians
    chunks: Tensor  # [B, K, 8] radians
    pad: Tensor  # [B, K] bool, True past the demo's end


def get_batch(part: Literal['train', 'val'], i: Tensor) -> Batch:
    # rows i of one part, every field at once; images to floats in [0, 1] only now
    def take(x) -> Tensor:  # x: a train/val pair from step 9's split
        return (x.train if part == 'train' else x.val)[i]

    return namespace(
        top=take(data.top).float() / 255,  # [B, 3, 128, 128]
        wrist=take(data.wrist).float() / 255,  # [B, 3, 128, 128]
        state=take(data.state),  # [B, 8]
        chunks=take(data.chunks),  # [B, K, 8] radians
        pad=take(data.pad),  # [B, K] bool
    )

In [20]:
@lambda f: f()
def probe():
    B, C, H, W = 2, 3, 4, 4
    pad = 2
    t = B * C * H * W
    x = torch.arange(t).reshape(B, C, H, W)
    show(x[0, 0])
    x = F.pad(x, (pad, pad, pad, pad), mode='replicate')
    show(x.shape, x[0, 0])
    ff = x[0, 0]
    for i in range(2 * pad + 1):
        r = ff[i : i + H, i : i + W]
        show(i, r)

x[0, 0] =
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15]])
x.shape = torch.Size([2, 3, 8, 8])
x[0, 0] =
tensor([[ 0,  0,  0,  1,  2,  3,  3,  3],
        [ 0,  0,  0,  1,  2,  3,  3,  3],
        [ 0,  0,  0,  1,  2,  3,  3,  3],
        [ 4,  4,  4,  5,  6,  7,  7,  7],
        [ 8,  8,  8,  9, 10, 11, 11, 11],
        [12, 12, 12, 13, 14, 15, 15, 15],
        [12, 12, 12, 13, 14, 15, 15, 15],
        [12, 12, 12, 13, 14, 15, 15, 15]])
i = 0
r =
tensor([[0, 0, 0, 1],
        [0, 0, 0, 1],
        [0, 0, 0, 1],
        [4, 4, 4, 5]])
i = 1
r =
tensor([[ 0,  0,  1,  2],
        [ 0,  0,  1,  2],
        [ 4,  4,  5,  6],
        [ 8,  8,  9, 10]])
i = 2
r =
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15]])
i = 3
r =
tensor([[ 5,  6,  7,  7],
        [ 9, 10, 11, 11],
        [13, 14, 15, 15],
        [13, 14, 15, 15]])
i = 4
r =
tensor([[10, 11, 11, 11],
        [14, 15, 15, 15

In [21]:
def flow_loss(net: VelocityNet, b: Batch, shift: int = 4) -> Tensor:
    # out: [], the toy's loss on chunks: MSE between predicted and true speed, padded steps skipped
    x1 = (b.chunks - stats.action.mean) / stats.action.std  # [B, K, 8], the demo chunk, std units
    x0 = torch.randn_like(x1)  # [B, K, 8], noise
    t = torch.rand(len(x1), device=dev)  # [B]
    xt = (1 - t[:, None, None]) * x0 + t[:, None, None] * x1  # [B, K, 8], the point at time t
    v = net(random_shift(b.top, shift), random_shift(b.wrist, shift), b.state, xt, t)
    err = (v - (x1 - x0)) ** 2  # [B, K, 8], MSE on purpose: its best guess is the mean speed
    keep = (~b.pad)[..., None].float()  # [B, K, 1], 0 on padded steps
    return (err * keep).sum() / (keep.sum() * 8)  # over real values only

In [22]:
torch.manual_seed(0)
b = get_batch('train', torch.randint(0, len(data.state.train), (64,), device=dev))
with torch.no_grad():
    loss = flow_loss(vnet, b)  # untrained: v ~ 0, so about E|x1 - x0|² = 1 + 1 = 2
show(b.pad.float().mean(), loss)

b.pad.float().mean() = tensor(0.0342, device='cuda:0')
loss = tensor(2.0624, device='cuda:0')


In [23]:
@torch.no_grad()
def sample(net: VelocityNet, top: Tensor, wrist: Tensor, state: Tensor, steps: int = 10) -> Tensor:
    # the toy's sampler (flow_matching.ipynb step 4), per observation
    # out: [B, K, 8] radians, one sampled plan per observation
    x = torch.randn(len(state), K, 8, device=dev)  # [B, K, 8], fresh noise
    for k in range(steps):
        t = torch.full((len(state),), k / steps, device=dev)  # [B]
        x = x + net(top, wrist, state, x, t) / steps  # Euler: x += speed * dt
    return x * stats.action.std + stats.action.mean  # std units -> radians


@torch.no_grad()
def val_error(net: VelocityNet) -> Tensor:
    # out: [8], mean |error| of one sampled plan's FIRST action per val frame, radians,
    # comparable with BC and ACT. 256 frames at a time, never shifted
    n = len(data.state.val)
    firsts = []
    for j in range(0, n, 256):
        b = get_batch('val', torch.arange(j, min(j + 256, n), device=dev))
        firsts.append(sample(net, b.top, b.wrist, b.state)[:, 0])  # [<=256, 8]
    return (torch.cat(firsts) - data.chunks.val[:, 0]).abs().mean(0)

In [24]:
@cache
def train(seed: int = 0, steps: int = 10000, lr: float = 1e-3) -> namespace:
    torch.manual_seed(seed)
    net = VelocityNet().to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=lr)
    err = torch.zeros(8)
    for step in range(steps + 1):
        b = get_batch('train', torch.randint(0, len(data.state.train), (64,), device=dev))
        loss = flow_loss(net, b)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % 2000 == 0:
            net.eval()
            err = val_error(net).cpu()
            net.train()
            print(
                f'{step:5d}  loss {loss.item():.3f}  val arm {err[:7].mean():.4f}  gripper {err[7]:.4f}'
            )
    # weights on the CPU, so the cache holds no GPU tensors
    return namespace(weights={k: v.cpu() for k, v in net.state_dict().items()}, err=err)

train: cached, 2 call(s) stored


In [25]:
def load(train_steps: int = 10000, log: bool = False) -> VelocityNet:
    # the net train(steps=train_steps) made, from its cache; log: replay its training log
    net = VelocityNet().to(dev)
    net.load_state_dict((train if log else train.quiet)(steps=train_steps).weights)
    return net.eval()


vnet = load(log=True)

    0  loss 2.057  val arm 0.1655  gripper 0.5478
 2000  loss 0.277  val arm 0.0266  gripper 0.1455
 4000  loss 0.302  val arm 0.0233  gripper 0.1266
 6000  loss 0.226  val arm 0.0202  gripper 0.1183
 8000  loss 0.181  val arm 0.0187  gripper 0.0957
10000  loss 0.188  val arm 0.0174  gripper 0.0909


### step 13 — the sim, and a rollout that samples a fresh plan every 8 steps

The repo's scheme: sample a plan, play 8 of its 16 steps blind, sample a fresh one. Not ACT's ensembling: averaging a left plan with a right plan gives a middle plan.

`Outcome` is a `NamedTuple`, as in act step 10: it unpacks as `ok, where` and reads as `.success`, `.where`.


In [26]:
import logging
import os

# render without a window; must be set before mujoco loads
os.environ.setdefault('MUJOCO_GL', 'egl')

logging.disable(logging.WARNING)  # robosuite warns while importing; mute that
import robosuite

logging.disable(logging.NOTSET)
logging.getLogger('robosuite_logs').setLevel(logging.ERROR)
from dream_robot.core.registry import make_env
from dream_robot.core.schema import Observation
from dream_robot.sims.robosuite.tasks.pick_place_two_bins.env import PickPlaceTwoBins

env: PickPlaceTwoBins = make_env(
    'robosuite/pick_place_two_bins_fixed'
)  # cube always at the centre, either bin wins

In [27]:
def frame(im: np.ndarray) -> Tensor:
    # im: [H, W, 3] uint8 from the sim
    # out: [1, 3, H, W] floats in [0, 1] on the GPU
    return torch.from_numpy(im).permute(2, 0, 1)[None].to(dev).float() / 255


def predict(net: VelocityNet, obs: Observation) -> NDArray[np.float32]:
    # out: [K, 8], one sampled plan from this observation; fresh noise each call
    st = torch.from_numpy(obs.state)[None].to(dev)  # [1, 8]
    plan = sample(net, frame(obs.images['top']), frame(obs.images['wrist']), st)[0].cpu().numpy()
    plan[:, 7] = np.clip(plan[:, 7], 0.0, 1.0)  # gripper commands must stay in [0, 1]
    return plan

In [28]:
from typing import NamedTuple

from dream_robot.core.video import observation_panel


class Outcome(NamedTuple):  # how one rollout ended
    success: bool
    where: str  # where the cube ended: 'left' / 'right' / 'between' / 'other'


def rollout(net: VelocityNet, seed: int, n_exec: int = 8, frames: list | None = None) -> Outcome:
    # sample a plan, play its first n_exec actions without looking, then sample again;
    # frames: if a list, one video frame per step is appended to it (wide view + both cameras)
    obs = env.reset(seed=seed)
    plan = []  # actions still to play from the last plan
    success = False
    for _ in range(600):
        if frames is not None:
            frames.append(observation_panel(env.render(), obs.images))
        if not plan:
            plan = list(predict(net, obs)[:n_exec])  # n_exec x [8]
        r = env.step(plan.pop(0))
        obs = r.observation
        if r.terminated or r.truncated:
            success = r.success
            break
    cube = env.cube_pos
    b = env.which_bin(cube)  # index into env.bin_names, or None
    where = (
        env.bin_names[b] if b is not None else ('between' if env.between_bins(cube) else 'other')
    )
    return Outcome(success, where)

In [29]:
torch.manual_seed(0)
ok, where = rollout(vnet, 1000)
show(ok, where)

ok = False
where = 'between'


### step 14 — the test that matters: one start, 20 times — does it use both bins? (~5 min the first time, cached after)

1. **same start**: env seed 1000 twenty times, only the net's noise changes. BC and ACT give the same bin 20/20 here by construction; the expert flips a coin.
2. **spread**: 20 different starts (seeds 1000-1019), the standard eval as for BC and ACT.

The cache keys on arguments, not on the global `vnet`, so each test names its net by `train_steps` and loads it with step 12's `load`.


In [30]:
from collections import Counter


def score(results: list[Outcome]) -> str:
    # 'wins/n   {where: count}'
    wins = sum(r.success for r in results)
    return f'{wins:2d}/{len(results)}   {dict(Counter(r.where for r in results))}'

In [31]:
@cache(2)
def same_start(train_steps: int = 10000, n_exec: int = 8) -> list[Outcome]:
    # does it use BOTH bins? One scene (env seed 1000) played 20 times; only the net's noise
    # differs (torch seed k). The expert flips a coin here, so the target is ~10 left, ~10 right.
    # BC and ACT are deterministic: the same bin 20/20 by construction
    net = load(train_steps)
    results = []
    for k in range(20):
        torch.manual_seed(k)
        results.append(rollout(net, 1000, n_exec))
    print(f'same start, {train_steps} steps, n_exec {n_exec}: {score(results)}')
    return results

same_start: cached, 4 call(s) stored


In [32]:
@cache(2)
def spread(train_steps: int = 10000, n_exec: int = 8) -> list[Outcome]:
    # does it WORK? The standard eval, as for BC and ACT: 20 different scenes (env seeds
    # 1000-1019), one rollout each, one torch seed for the whole run. The cube is always at the
    # centre in T2-fixed, so the scenes differ only in the start's small jitter
    net = load(train_steps)
    torch.manual_seed(0)
    results = [rollout(net, seed, n_exec) for seed in range(1000, 1020)]
    print(f'spread,     {train_steps} steps, n_exec {n_exec}: {score(results)}')
    return results

spread: cached, 4 call(s) stored


In [33]:
same = same_start()
spr = spread()

same start, 10000 steps, n_exec 8:  5/20   {'between': 15, 'right': 3, 'left': 2}
spread,     10000 steps, n_exec 8:  7/20   {'between': 12, 'right': 6, 'left': 1, 'other': 1}


### step 15 — play all 16 steps of each plan, not 8 (~5 min the first time, cached after)

Step 14's tests on the same net, but each plan is played to its end before sampling the next: half as many fresh samples per episode, so half the chances to switch bins mid-carry.


In [34]:
same16 = same_start(n_exec=16)
spr16 = spread(n_exec=16)

same start, 10000 steps, n_exec 16: 17/20   {'right': 9, 'left': 8, 'between': 3}
spread,     10000 steps, n_exec 16: 15/20   {'right': 10, 'between': 5, 'left': 5}


**Result (steps 14-15)**, the 10k-step net:

| plays per plan | same start: wins, where           | spread: wins, where                        |
| -------------- | --------------------------------- | ------------------------------------------ |
| 8 of 16        | 5/20: between 15, right 3, left 2 | 7/20: between 12, right 6, left 1, other 1 |
| 16 of 16       | 17/20: right 9, left 8, between 3 | 15/20: right 10, left 5, between 5         |

- It does use both bins from one scene: 9 right / 8 left at 16, where BC and ACT can't.
- Playing the whole plan: 5 → 17 and 7 → 15, almost all of it fewer `'between'`.
- `'between'` is any cube resting between the bins, so a cube never carried away from the centre counts too. The filmed 16-step one (below, k=6) is that: the grasp at step ~80 tilts the cube, it slips out by ~100, lands back at the centre, and the arm never re-grasps it in 600 steps.
- The filmed 8-step one (k=0) is the same, not a bin switch: grasp at ~80, cube lifted tilted at ~88, dropped by ~96, back at the centre; the arm then wanders (odd sideways poses late) and never re-grasps. So far it's the grasp, not the coin, that fails; one film each.
- Close to `flow_matching.ipynb` (11 and 8 at 8 steps, 17 and 16 at 16); the 8-step runs are noisier.


### a 16-step failure, filmed: the cube dropped between the bins

The first `'between'` run of `same_start(n_exec=16)`, replayed with its torch seed; `rollout` appends a frame per step.


In [35]:
from IPython.display import Video, display

from dream_robot.core.video import write_video


def film(results: list[Outcome], n_exec: int, where: str = 'between', j: int = 0) -> None:
    # replay same_start's j-th run that ended at `where`, filmed: same env seed, same torch seed
    ks = [k for k, r in enumerate(results) if r.where == where]  # torch seeds that ended there
    assert j < len(ks), 'select lower j'
    k = ks[j]
    frames = []  # [H, W, 3] each: wide view + the policy's two cameras
    torch.manual_seed(k)
    replay = rollout(vnet, 1000, n_exec, frames=frames)
    show(ks, k, replay, len(frames))
    path = REPO / f'scratch/flow_matching_namespace/{where}_n{n_exec}_k{k}.mp4'
    video = write_video(frames, path, fps=env.control_hz)
    display(Video(video, embed=True, width=720))  # embedded, so it plays without the file

In [36]:
film(same16, n_exec=16, j=2)

ks = [6, 13, 14]
k = 14
replay = Outcome(success=False, where='between')
len(frames) = 600


### an 8-step failure, filmed: bin switch mid-carry, or the same slipped grasp?

The same slipped grasp (k=0): lifted tilted, dropped by step ~96, never re-grasped.


In [37]:
film(same, n_exec=8, j=2)

ks = [0, 3, 4, 6, 7, 8, 10, 12, 13, 14, 15, 16, 17, 18, 19]
k = 4
replay = Outcome(success=False, where='between')
len(frames) = 600


### step 16 — train to 30k steps, then both tests at 8 and 16

30k steps is about the repo's 100 epochs. A fresh run from seed 0, not step 12's net trained on: the cache keys each net by its arguments, so `train(steps=30000)` is its own entry (~13 min the first time). Same seed and no learning-rate schedule, so its first 10k steps are step 12's run again: the log up to 10000 should match step 12's (up to GPU noise), and the net at 30k is step 12's net trained 20k more, as in `flow_matching.ipynb`.


In [38]:
vnet30 = load(30000, log=True)

    0  loss 2.057  val arm 0.1655  gripper 0.5478
 2000  loss 0.277  val arm 0.0262  gripper 0.1466
 4000  loss 0.303  val arm 0.0233  gripper 0.1253
 6000  loss 0.227  val arm 0.0202  gripper 0.1202
 8000  loss 0.183  val arm 0.0184  gripper 0.0971
10000  loss 0.190  val arm 0.0175  gripper 0.0912
12000  loss 0.207  val arm 0.0172  gripper 0.0872
14000  loss 0.225  val arm 0.0150  gripper 0.0835
16000  loss 0.190  val arm 0.0148  gripper 0.0747
18000  loss 0.166  val arm 0.0138  gripper 0.0734
20000  loss 0.207  val arm 0.0137  gripper 0.0701
22000  loss 0.160  val arm 0.0129  gripper 0.0679
24000  loss 0.160  val arm 0.0130  gripper 0.0663
26000  loss 0.178  val arm 0.0131  gripper 0.0620
28000  loss 0.142  val arm 0.0130  gripper 0.0591
30000  loss 0.219  val arm 0.0118  gripper 0.0618


In [39]:
# steps 14 and 15's tests on the 30k net (~20 min the first time, cached after)
for n_exec in [8, 16]:
    same_start(30000, n_exec)
    spread(30000, n_exec)

same start, 30000 steps, n_exec 8: 19/20   {'left': 17, 'right': 2, 'other': 1}
spread,     30000 steps, n_exec 8: 18/20   {'left': 15, 'right': 3, 'other': 1, 'between': 1}
same start, 30000 steps, n_exec 16: 20/20   {'left': 17, 'right': 3}
spread,     30000 steps, n_exec 16: 18/20   {'left': 14, 'right': 4, 'between': 1, 'other': 1}


**Result (step 16)**, the 30k-step net (same seed, so its first 10k matched step 12's log to ~0.0003):

| plays per plan | same start: wins, where          | spread: wins, where                         |
| -------------- | -------------------------------- | ------------------------------------------- |
| 8 of 16        | 19/20: left 17, right 2, other 1 | 18/20: left 15, right 3, other 1, between 1 |
| 16 of 16       | 20/20: left 17, right 3          | 18/20: left 14, right 4, between 1, other 1 |

- Training longer fixed the grasp: `'between'` 15 → 0 (same start, 8) and 3 → 0 (16). The slipped grasp was undertraining, and 8 vs 16 hardly matters now.
- But the coin is now lopsided: 17 left / 3 right from one scene, at both 8 and 16. A fair coin gives 17+ of one side in 20 about 0.3% of the time. `flow_matching.ipynb`'s 30k net split 13/7 and 9/11.
- So longer training traded the even split for success, on this seed. One seed only: a second seed would tell whether the lean is the method or this run.
